# Supercooling in the Model Output

The degree of supercooling is given by

$$ \Delta T_{SC} = T_F - T $$


We distinguish between 
- **model potential supercooling**: with $T_F$ as the (constant) model freezing point -1.9
- **physical potential supercooling**: with $T=T_{pot}$ and $T_F = T_F(p=0,S)$ (physical surface-referenced freezing point)
- **in-situ supercooling**: with $T=T_{in-situ}$ and $T_F = T_F(p,S)$ (physical in-situ freezing point)

## Import Packages & Load Data

In [1]:
# packages for data processing
import numpy as np
import pandas as pd
import xarray as xr
from xmitgcm import open_mdsdataset
import gsw as gsw

# packages for plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go # interactive/3D plotting
import plotly.express as px
import cmocean.cm as cmo # ocean colormaps

# my own plotting functions
from plotting_functions import *

# technical packages
import warnings

In [24]:
model_run = "MSL007"
delta_t = 4 # time step in seconds
output_dir = "../../MITgcm/so_plumes/" + model_run

# surpress the xmitgcm warning about future changes with timedelta
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)

    # open dataset
    ds = open_mdsdataset(output_dir, prefix = ['Eta', 'U', 'W', 'T', 'V', 'S', 'PH'], delta_t = delta_t, geometry = "cartesian")
print(ds["T"].min().values)

-2.0005


In [25]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:  (XC: 594, YC: 66, XG: 594, YG: 66, Z: 99, Zp1: 100, Zu: 99,
              Zl: 99, time: 19)
Coordinates: (12/34)
  * XC       (XC) >f4 2kB 2.0 6.0 10.0 14.0 ... 2.366e+03 2.37e+03 2.374e+03
  * YC       (YC) >f4 264B 2.0 6.0 10.0 14.0 18.0 ... 250.0 254.0 258.0 262.0
  * XG       (XG) >f4 2kB 0.0 4.0 8.0 12.0 ... 2.364e+03 2.368e+03 2.372e+03
  * YG       (YG) >f4 264B 0.0 4.0 8.0 12.0 16.0 ... 248.0 252.0 256.0 260.0
  * Z        (Z) >f4 396B -0.5 -1.5 -2.5 -3.5 ... -360.0 -370.0 -380.0 -390.5
  * Zp1      (Zp1) >f4 400B 0.0 -1.0 -2.0 -3.0 ... -365.0 -375.0 -385.0 -396.0
    ...       ...
    dxV      (YG, XG) >f4 157kB dask.array<chunksize=(66, 594), meta=np.ndarray>
    rhoRef   (Z) >f4 396B dask.array<chunksize=(99,), meta=np.ndarray>
    dyF      (YC, XC) >f4 157kB dask.array<chunksize=(66, 594), meta=np.ndarray>
    dyU      (YG, XG) >f4 157kB dask.array<chunksize=(66, 594), meta=np.ndarray>
    dxF      (YC, XC) >f4 157kB dask.array<chunksize=(66, 594), meta=np.ndarray>
    iter     (time) int64 152B dask.array<chunksize=(1,), meta=np.ndarray>
Data variables:
    Eta      (time, YC, XC) float32 3MB dask.array<chunksize=(1, 66, 594), meta=np.ndarray>
    S        (time, Z, YC, XC) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
    T        (time, Z, YC, XC) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
    W        (time, Zl, YC, XC) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
    V        (time, Z, YG, XC) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
    U        (time, Z, YC, XG) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
    PH       (time, Z, YC, XC) float32 295MB dask.array<chunksize=(1, 99, 66, 594), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    title:        netCDF wrapper of MITgcm MDS binary data
    source:       MITgcm
    history:      Created by calling `open_mdsdataset(data_dir='../../MITgcm/...

## Model Potential Supercooling

To calculate the model potential supercooling we only need the (constant) model freezing point `Tfreezing` and the potential temperature, which is already included in the model output.

In [26]:
Tfreezing = -1.9
ds["modelSC"] = (Tfreezing-ds["T"])
ds["modelSC"].attrs["standard_name"] = "model_supercooling"
ds["modelSC"].attrs["long_name"] = "Model Supercooling"
ds["modelSC"].attrs["units"] = "degrees_Celsius"

## Physical Potential Supercooling

To calculate the physical potential supercooling we need the surface freezing point and the potential temperature, which is already included in the model output.

In [27]:
Tfreezing_p0 = gsw.t_freezing(ds["S"], 0, 0)
ds["potSC"] = (Tfreezing_p0-ds["T"])
ds["potSC"].attrs["standard_name"] = "potential_supercooling"
ds["potSC"].attrs["long_name"] = "Potential Supercooling"
ds["potSC"].attrs["units"] = "degrees_Celsius"


## In-Situ Supercooling

To calculate the in-situ supercooling we need to first calculate
- the in situ temperature
- the in-situ freezing point

In-situ temperature:



In [28]:
# conservative temperature
ds["CT"] = gsw.CT_from_pt(ds["S"], ds["T"])
# calculate sea pressure
ds["p"] = ds["PH"] + ds["PHrefC"]
ds["T_situ"]=gsw.t_from_CT(ds["S"], ds["CT"], ds["p"])

In [29]:
# freezing point
ds["T_f"] = gsw.t_freezing(ds["S"], ds["p"], 0) # what about oxygen saturation fraction?

# in situ supercooling
ds["insituSC"] = ds["T_f"] - ds["T_situ"]
ds["insituSC"].attrs["standard_name"] = "in_situ_supercooling"
ds["insituSC"].attrs["long_name"] = "In Situ Supercooling"
ds["insituSC"].attrs["units"] = "degrees_Celsius"

compare with

`gsw.t_freezing_poly(SA, p, saturation_fraction)` (Polynomial)

## Plot Lead View

In [30]:
# crop to lead view
x_min = 0.0
x_max = 1190.0
mask = (ds["XC"] >= x_min) & (ds["XC"] <= x_max)
ds_sub = ds.where(mask, drop=True)

In [22]:
print("Max. model supercooling: ", ds_sub["model_SC"].max().values)
print("Max. potential supercooling: ", ds_sub["potSC"].max().values)
print("Max. in situ supercooling: ", ds_sub["insitu_SC"].max().values)

Max. model supercooling:  0.10049999
Max. potential supercooling:  0.136484460326731
Max. in situ supercooling:  0.13263979508605583


In [31]:
var = "insituSC"
print("animation_" + model_run + "_" + var + "_" + ".html")

animation_MSL007_insituSC_.html


In [33]:
# monodirectional colormap (we set cmin=0 for this)
var = "insituSC"
fig = cube_time_evol(ds_sub, var, model_run, colorscale="ice_r", grid=False)

filename = "animation_" + model_run + "_" + var + "_" + ".html"
fig.write_html(filename)
print(filename)
# fig.show(renderer="browser") # works in vscode, not in jupyterhub

In [ ]:
# diverging colormap (this forces cmin=cmax)
fig = cube_time_evol(ds_sub, "model_SC", model_run, colorscale="balance_r", grid=False)
# fig.show(renderer="browser") # works in vscode, not in jupyterhub

(node:3613389) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)


## Depth Statistics

I want to create a plot that shows 
- the percentage of supercooled cells
- the average degree of supercooling withing those

for each depth within the lead.